# Chương 18: Đạo đức và Quản trị Trí tuệ Nhân tạo

Notebook này triển khai thực nghiệm 2 thuật toán căn chỉnh mô hình ngôn ngữ lớn (LLM Alignment) hiện đại nhất:
1. **Mô hình phần thưởng Bradley-Terry (Reward Model)**: Cài đặt hàm mất mát xếp hạng ưu tiên (pairwise ranking loss) của con người và huấn luyện mô hình dự đoán điểm thưởng.
2. **Hàm mất mát Tối ưu hóa ưu tiên trực tiếp (Direct Preference Optimization - DPO)**: Cài đặt DPO Loss từ đầu bằng NumPy, tính toán đạo hàm giải tích và mô phỏng quá trình điều chỉnh xác suất chính sách ngôn ngữ $\pi_\theta$ tăng điểm cho câu trả lời tốt ($y_w$) và phạt câu trả lời xấu ($y_l$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Mô hình phần thưởng Bradley-Terry (Reward Model)

### Cơ sở toán học
Cho cặp câu trả lời $(y_w, y_l)$ trong đó $y_w$ được con người đánh giá cao hơn $y_l$ ($y_w \succ y_l$).
Xác suất theo mô hình Bradley-Terry:
$$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l)) = \frac{1}{1 + e^{-(r(x, y_w) - r(x, y_l))}}$$
Hàm mất mát cực tiểu hóa âm log-likelihood:
$$\mathcal{L}_{\text{RM}} = - \frac{1}{N} \sum_{i=1}^N \log \sigma(r_i^{(w)} - r_i^{(l)})$$

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def reward_model_loss(r_w, r_l):
    """
    r_w: điểm thưởng dự đoán cho câu trả lời thắng cuộc
    r_l: điểm thưởng dự đoán cho câu trả lời thua cuộc
    """
    margin = r_w - r_l
    probs = sigmoid(margin)
    loss = -np.mean(np.log(probs + 1e-12))
    # Đạo hàm dL/d(margin) = -(1 - probs)
    grad_margin = -(1.0 - probs) / len(r_w)
    return loss, grad_margin

# Mô phỏng một mô hình phần thưởng tuyến tính r(x, y) = w^T f(x, y)
# 50 cặp câu trả lời với vector đặc trưng chất lượng (ví dụ: độ chính xác thông tin, tính hữu ích, tính an toàn)
num_pairs = 60
feat_dim = 4

# Đặc trưng thực: y_w có đặc trưng tốt hơn y_l
feat_w = np.random.randn(num_pairs, feat_dim) + 0.5
feat_l = np.random.randn(num_pairs, feat_dim) - 0.5

w_reward = np.zeros(feat_dim)
lr = 0.1
rm_losses = []

for epoch in range(100):
    r_w = feat_w @ w_reward
    r_l = feat_l @ w_reward
    loss, grad_margin = reward_model_loss(r_w, r_l)
    rm_losses.append(loss)
    
    # Lan truyền ngược cập nhật trọng số thưởng
    # dL/dw = sum grad_margin * (feat_w - feat_l)
    grad_w = (feat_w - feat_l).T @ grad_margin
    w_reward -= lr * grad_w

final_acc = np.mean((feat_w @ w_reward) > (feat_l @ w_reward)) * 100
print(f"Mất mát ban đầu: {rm_losses[0]:.4f} -> Mất mát sau 100 bước: {rm_losses[-1]:.4f}")
print(f"Độ chính xác xếp hạng của Reward Model: {final_acc:.2f}%")

## 2. Hàm mất mát Tối ưu hóa ưu tiên trực tiếp (Direct Preference Optimization - DPO)

### Công thức toán học (Rafailov et al., 2023)
$$\mathcal{L}_{\text{DPO}}(\theta; \pi_{\text{ref}}) = - \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right)$$

Hàm phần thưởng tiềm ẩn ngầm định (Implicit Reward):
$$\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

In [ ]:
def compute_dpo_loss(pi_logps_w, pi_logps_l, ref_logps_w, ref_logps_l, beta=0.1):
    """
    pi_logps_w: log xác suất của câu trả lời thắng cuộc theo mô hình chính sách pi_theta
    pi_logps_l: log xác suất của câu trả lời thua cuộc theo mô hình chính sách pi_theta
    ref_logps_w: log xác suất của câu trả lời thắng cuộc theo mô hình tham chiếu pi_ref (đóng băng)
    ref_logps_l: log xác suất của câu trả lời thua cuộc theo mô hình tham chiếu pi_ref (đóng băng)
    beta: hệ số phạt phân kỳ KL
    """
    # Tính log-ratio giữa chính sách hiện tại và mô hình tham chiếu
    pi_ratio_w = pi_logps_w - ref_logps_w
    pi_ratio_l = pi_logps_l - ref_logps_l
    
    # Điểm thưởng tiềm ẩn (Implicit Rewards)
    implicit_reward_w = beta * pi_ratio_w
    implicit_reward_l = beta * pi_ratio_l
    
    # Độ chênh lệch điểm thưởng tiềm ẩn (Margin)
    reward_margins = implicit_reward_w - implicit_reward_l
    
    # DPO Loss
    losses = -np.log(sigmoid(reward_margins) + 1e-12)
    loss = np.mean(losses)
    
    # Trọng số gradient cập nhật của DPO: sigmoid(implicit_reward_l - implicit_reward_w)
    grad_weights = sigmoid(-reward_margins)
    
    return loss, implicit_reward_w, implicit_reward_l, grad_weights

# Mô phỏng 1 batch gồm 5 mẫu đối thoại
B = 5
# Mô hình tham chiếu ban đầu có xác suất xấp xỉ ngang nhau giữa y_w và y_l
ref_logps_w = np.full(B, -1.2)
ref_logps_l = np.full(B, -1.2)

# Mô hình chính sách pi_theta chưa căn chỉnh
pi_logps_w = np.copy(ref_logps_w)
pi_logps_l = np.copy(ref_logps_l)

dpo_losses = []
lr_policy = 0.5

# Quá trình tối ưu hóa DPO trực tiếp
for step in range(40):
    loss, rew_w, rew_l, weights = compute_dpo_loss(
        pi_logps_w, pi_logps_l, ref_logps_w, ref_logps_l, beta=0.2
    )
    dpo_losses.append(loss)
    
    # Gradient của DPO thúc đẩy: tăng pi_logps_w và giảm pi_logps_l
    pi_logps_w += lr_policy * (0.2 * weights)
    pi_logps_l -= lr_policy * (0.2 * weights)

print(f"DPO Loss ban đầu: {dpo_losses[0]:.4f} -> Sau 40 bước: {dpo_losses[-1]:.4f}")
print(f"Xác suất câu trả lời thắng cuộc P_theta(y_w): {np.exp(pi_logps_w[0]):.4f} (Tăng vọt từ {np.exp(ref_logps_w[0]):.4f})")
print(f"Xác suất câu trả lời thua cuộc P_theta(y_l):  {np.exp(pi_logps_l[0]):.4f} (Bị triệt tiêu xuống từ {np.exp(ref_logps_l[0]):.4f})")

### Trực quan hóa mặt hàm mất mát và quỹ đạo hội tụ của DPO

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 1. Đường cong hội tụ DPO Loss
ax1.plot(dpo_losses, color="indigo", lw=2, label="DPO Loss")
ax1.set_title("Quá trình cực tiểu hóa Direct Preference Optimization")
ax1.set_xlabel("Số bước cập nhật (Steps)")
ax1.set_ylabel("Loss")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# 2. Diễn biến điểm thưởng tiềm ẩn (Implicit Reward)
margins = np.linspace(-3, 3, 200)
losses_curve = -np.log(sigmoid(margins))
grad_curve = sigmoid(-margins)

ax2.plot(margins, losses_curve, color="crimson", lw=2, label="DPO Loss: -log(sigma(margin))")
ax2.plot(margins, grad_curve, color="teal", lw=2, linestyle="--", label="Hệ số Gradient: sigma(-margin)")
ax2.set_title("Hàm mất mát và trọng số gradient theo Margin thưởng ngầm")
ax2.set_xlabel("Reward Margin: $\beta \cdot [\log(\pi/\pi_{ref})_w - \log(\pi/\pi_{ref})_l]$")
ax2.set_ylabel("Giá trị")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

print("Hoàn tất toàn bộ thực hành Chapter 18: Đã kiểm chứng trọn vẹn Bradley-Terry Reward Model và DPO Loss!")